In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported!')

✅ Libraries imported!


In [2]:
df = pd.read_csv('../data/tamil_nadu_waterborne_disease_data.csv')
print(f'✅ Data loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(3)

✅ Data loaded: 1008 rows, 19 columns


,date,year,month,month_name,district,rainfall_mm,temperature_celsius,pH,turbidity_NTU,coliform_count_per100ml,dissolved_oxygen_mg_L,total_dissolved_solids_mg_L,population,cholera_cases,diarrhea_cases,typhoid_cases,hepatitis_a_cases,total_disease_cases,outbreak_alert
0,2018-01-01,2018,1,January,Chennai,36.22,37.31,7.88,15.37,78.01,4.78,258.08,4544887,42,170,26,15,253,1
1,2018-02-01,2018,2,February,Chennai,13.95,34.11,8.36,1.02,496.11,7.09,811.65,1807371,62,243,44,20,369,1
2,2018-03-01,2018,3,March,Chennai,78.16,27.26,6.41,15.84,191.23,8.92,666.76,2928388,48,176,32,13,269,1


In [3]:
def get_season(month):
    if month in [10, 11, 12]:
        return 'NE_Monsoon'
    elif month in [6, 7, 8, 9]:
        return 'SW_Monsoon'
    elif month in [3, 4, 5]:
        return 'Summer'
    else:
        return 'Winter'

df['season'] = df['month'].apply(get_season)
print('✅ Season column added!')
print(df['season'].value_counts())

✅ Season column added!
season
SW_Monsoon    336
Summer        252
NE_Monsoon    252
Winter        168
Name: count, dtype: int64


In [4]:
df = df.sort_values(['district', 'date']).reset_index(drop=True)

df['cases_last_month'] = df.groupby('district')[
    'total_disease_cases'].shift(1)
df['cases_2_months_ago'] = df.groupby('district')[
    'total_disease_cases'].shift(2)
df['rainfall_last_month'] = df.groupby('district')[
    'rainfall_mm'].shift(1)

df['cases_last_month'] = df['cases_last_month'].fillna(0)
df['cases_2_months_ago'] = df['cases_2_months_ago'].fillna(0)
df['rainfall_last_month'] = df['rainfall_last_month'].fillna(0)

print('✅ Lag features added!')
print(df[['district', 'date', 'total_disease_cases',
          'cases_last_month', 'cases_2_months_ago']].head(8))

✅ Lag features added!
  district        date  total_disease_cases  cases_last_month  \
0  Chennai  2018-01-01                  253               0.0   
1  Chennai  2018-02-01                  369             253.0   
2  Chennai  2018-03-01                  269             369.0   
3  Chennai  2018-04-01                  231             269.0   
4  Chennai  2018-05-01                  200             231.0   
5  Chennai  2018-06-01                  556             200.0   
6  Chennai  2018-07-01                  288             556.0   
7  Chennai  2018-08-01                  330             288.0   

   cases_2_months_ago  
0                 0.0  
1                 0.0  
2               253.0  
3               369.0  
4               269.0  
5               231.0  
6               200.0  
7               556.0  


In [5]:
le_district = LabelEncoder()
df['district_encoded'] = le_district.fit_transform(df['district'])

le_season = LabelEncoder()
df['season_encoded'] = le_season.fit_transform(df['season'])

print('✅ Encoding done!')
print('\nDistrict Encoding:')
for i, district in enumerate(le_district.classes_):
    print(f'  {district} → {i}')
print('\nSeason Encoding:')
for i, season in enumerate(le_season.classes_):
    print(f'  {season} → {i}')

✅ Encoding done!

District Encoding:
  Chennai → 0
  Coimbatore → 1
  Cuddalore → 2
  Madurai → 3
  Nagapattinam → 4
  Ramanathapuram → 5
  Salem → 6
  Thanjavur → 7
  Tiruchirappalli → 8
  Tirunelveli → 9
  Tiruppur → 10
  Vellore → 11

Season Encoding:
  NE_Monsoon → 0
  SW_Monsoon → 1
  Summer → 2
  Winter → 3


In [6]:
feature_cols = [
    'rainfall_mm',
    'temperature_celsius',
    'pH',
    'turbidity_NTU',
    'coliform_count_per100ml',
    'dissolved_oxygen_mg_L',
    'total_dissolved_solids_mg_L',
    'month',
    'district_encoded',
    'season_encoded',
    'cases_last_month',
    'cases_2_months_ago',
    'rainfall_last_month'
]

target_col = 'outbreak_alert'

X = df[feature_cols]
y = df[target_col]

print(f'✅ Features selected: {len(feature_cols)}')
print(f'📊 X shape: {X.shape}')
print(f'🎯 y shape: {y.shape}')

✅ Features selected: 13
📊 X shape: (1008, 13)
🎯 y shape: (1008,)


In [7]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

print('✅ Features scaled successfully!')
print('\nSample scaled values:')
print(X_scaled.head(3).round(3))

✅ Features scaled successfully!

Sample scaled values:
   rainfall_mm  temperature_celsius     pH  turbidity_NTU  \
0       -0.965                1.629  0.767         -0.142   
1       -1.186                0.835  1.498         -1.595   
2       -0.548               -0.865 -1.472         -0.095   

   coliform_count_per100ml  dissolved_oxygen_mg_L  \
0                   -1.164                 -1.145   
1                    0.871                  0.444   
2                   -0.613                  1.703   

   total_dissolved_solids_mg_L  month  district_encoded  season_encoded  \
0                       -1.598 -1.593            -1.593           1.622   
1                        0.324 -1.304            -1.593           1.622   
2                       -0.179 -1.014            -1.593           0.649   

   cases_last_month  cases_2_months_ago  rainfall_last_month  
0            -3.071              -2.896               -1.299  
1            -0.880              -2.896               -0.936

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('✅ Train-Test Split Done!')
print(f'\n📊 Training set: {X_train.shape[0]} rows')
print(f'📊 Testing set:  {X_test.shape[0]} rows')
print(f'\n🎯 Training labels:')
print(y_train.value_counts())
print(f'\n🎯 Testing labels:')
print(y_test.value_counts())

✅ Train-Test Split Done!

📊 Training set: 806 rows
📊 Testing set:  202 rows

🎯 Training labels:
outbreak_alert
1    806
Name: count, dtype: int64

🎯 Testing labels:
outbreak_alert
1    202
Name: count, dtype: int64


In [9]:
import os
os.makedirs('../models', exist_ok=True)

df.to_csv('../data/processed_data.csv', index=False)
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(le_district, '../models/le_district.pkl')
joblib.dump(le_season, '../models/le_season.pkl')

print('✅ All files saved!')
print('\n📁 Saved to data/:')
print('  - processed_data.csv')
print('  - X_train.csv, X_test.csv')
print('  - y_train.csv, y_test.csv')
print('\n🤖 Saved to models/:')
print('  - scaler.pkl')
print('  - le_district.pkl')
print('  - le_season.pkl')

✅ All files saved!

📁 Saved to data/:
  - processed_data.csv
  - X_train.csv, X_test.csv
  - y_train.csv, y_test.csv

🤖 Saved to models/:
  - scaler.pkl
  - le_district.pkl
  - le_season.pkl


In [10]:
print('='*50)
print('📊 DAY 6 PREPROCESSING SUMMARY')
print('='*50)
print(f'✅ Total features used: {len(feature_cols)}')
print(f'✅ Training samples: {X_train.shape[0]}')
print(f'✅ Testing samples: {X_test.shape[0]}')
print(f'✅ Outbreak cases in train: {y_train.sum()}')
print(f'✅ Scaler + Encoders saved!')
print('\n🚀 Ready for ML Model Building tomorrow!')

📊 DAY 6 PREPROCESSING SUMMARY
✅ Total features used: 13
✅ Training samples: 806
✅ Testing samples: 202
✅ Outbreak cases in train: 806
✅ Scaler + Encoders saved!

🚀 Ready for ML Model Building tomorrow!
